# Structured Outputs, Output Guardrails, and Context7 MCP

This notebook demonstrates three complementary patterns in the OpenAI Agents SDK:

1. Returning validated structured data from an agent with a Pydantic model.
2. Applying an output guardrail that can stop unsuitable agent responses.
3. Connecting an agent to the Context7 MCP server for documentation-grounded answers.

The examples use Gemini through an OpenAI-compatible endpoint while orchestration is handled by the OpenAI Agents SDK.


## Prerequisites

Install the required packages and create a `.env` file containing:

```text
GOOGLE_API_KEY=your_google_api_key
```

The notebook expects an environment that includes packages such as `openai-agents`, `openai`, `pydantic`, `python-dotenv`, and `mcp`.

> Do not commit the `.env` file or API keys to Git.


## 1. Imports and environment setup

Load environment variables and import the SDK, validation, email, and HTTP utilities used by the examples.


In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field
from email.message import EmailMessage
import smtplib
import requests
load_dotenv(override=True)

## 2. Configure Gemini as the agent model

The Google Gemini OpenAI-compatible endpoint is wrapped with `AsyncOpenAI`, then passed to `OpenAIChatCompletionsModel`.

The API-key check only confirms that a value was loaded; it does not print the full secret.


In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)

## 3. Structured output with Pydantic

`EmailReview` defines the exact schema expected from the reviewing agent. Instead of returning unstructured prose, the agent must populate validated fields describing the email.


In [ ]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

Inspect the JSON Schema generated by Pydantic. This is the schema supplied to the model for structured output generation.


In [ ]:
EmailReview.model_json_schema()

### Example email

The sample intentionally contains an informal tone and a personalization placeholder so that the reviewer has issues to detect.


In [ ]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Bye.
Deepak
"""

### Run the structured-output reviewer

Setting `output_type=EmailReview` instructs the SDK to parse and validate the final response as an `EmailReview` object.


In [ ]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=gemini_model, output_type=EmailReview)
result = await Runner.run(checker, email)

### Inspect the validated result

The result is a typed Pydantic object, so individual fields can be accessed directly.


In [ ]:
review = result.final_output
review

In [ ]:
review.is_professional

## 4. Output guardrail

The guardrail reuses the checker agent to review another agent's final response.

The tripwire is triggered when either condition is true:

- the response still contains placeholders;
- the response is not professional.

A triggered tripwire prevents the output from being accepted as a successful final response.


In [ ]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

## 5. Agent instructions and test agent

The following text provides reusable sales-manager instructions and a larger task definition. In this notebook, the guardrail demonstration uses the cowboy-style agent below; the longer orchestration task is retained as a foundation for a multi-agent extension.


In [ ]:
instructions = """
You are a Sales Manager at DEEPLABAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""



In [ ]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

### Run the agent and handle a guardrail tripwire

`OutputGuardrailTripwireTriggered` is caught explicitly so the notebook can display:

- which guardrail fired;
- the rejected agent output;
- the structured review details;
- the final tripwire status.


In [ ]:
from agents import OutputGuardrailTripwireTriggered

try:
    result = await Runner.run(
        sales_agent_cowboy,
        "Write a cold sales email",
    )
    print(result.final_output)

except OutputGuardrailTripwireTriggered as exc:
    guardrail_result = exc.guardrail_result

    print("Output guardrail triggered")
    print("Guardrail:", guardrail_result.guardrail.get_name())
    print("Agent output:", guardrail_result.agent_output)
    print("Guardrail details:", guardrail_result.output.output_info)
    print(
        "Tripwire triggered:",
        guardrail_result.output.tripwire_triggered,
    )

## 6. Context7 through MCP

This section connects the agent to the remote Context7 MCP server. Context7 supplies tools that allow the model to resolve a library and query its current documentation.


In [ ]:
from agents.mcp import MCPServerStreamableHttp

### Use Gemini 2.5 Flash for MCP tool calling

This section uses `gemini-2.5-flash`, which is compatible with the tool-calling flow used in this example.


In [ ]:
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=gemini_client,
)

### Ask a documentation-grounded question

`MCPServerStreamableHttp` manages the remote MCP session.

Two timeout settings serve different purposes:

- `params["timeout"]` configures the underlying HTTP transport.
- `client_session_timeout_seconds` controls how long the MCP client waits for operations such as `query-docs`.

Retries are enabled for transient MCP failures. The agent is explicitly instructed to use Context7 before answering rather than relying only on its internal knowledge.


In [ ]:
task = """
In the new Sandbox Agents feature in the OpenAI Agents SDK as of May 2026,
what is the role of the Manifest object?

Use Context7 to consult the OpenAI Agents SDK documentation before answering.
If the documentation does not support the answer, say so.
"""

params = {
    "url": "https://mcp.context7.com/mcp",
    "timeout": 60,
}

async with MCPServerStreamableHttp(
    name="Context7",
    params=params,
    client_session_timeout_seconds=60,
    max_retry_attempts=2,
    retry_backoff_seconds_base=1.0,
) as server:
    agent = Agent(
        name="Expert",
        instructions=(
            "Always use Context7 before answering. "
            "Search the relevant official library documentation."
        ),
        mcp_servers=[server],
        model=gemini_model,
    )

    result = await Runner.run(agent, task)

print(result.final_output)


## Key takeaways

- Pydantic models make agent outputs predictable, typed, and easy to validate.
- Output guardrails can inspect a completed response and stop it with a tripwire.
- A guardrail can itself call another agent for model-based evaluation.
- MCP gives agents access to external tools without hard-coding each tool into the workflow.
- Transport timeouts and MCP request timeouts are separate settings.
- Documentation-grounded answers are more reliable for recent or version-specific SDK features.
